# Pipeline de Carga, Diagnóstico y Preparación de Datos: Pokémon Gen 1-9

Este notebook contiene el flujo de trabajo para la adquisición, análisis descriptivo preliminar y transformación de los datos del dataset de clasificación de Pokémon (Generaciones 1 a 9). El objetivo es modularizar los componentes clave en scripts de Python reutilizables y preparar los datos estructurados para su posterior uso en tareas de análisis o modelado.

Carga y Creación de Dataset

In [ ]:
%pip install datasets pandas

   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/604.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/604.4 kB ? eta -:--:--
   ---------------------------------------- 604.4/604.4 kB 750.3 kB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ------------ --------------------------- 262.1/857.0 kB ? eta -:--:--
   ------------------------ --------------- 524.3/857.0 kB 1.1 MB/s eta 0:00:01
   ------------------------ --------------- 524.3/857.0 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 857.0/857.0 kB 889.6 kB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   -- -----------------------------------


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Módulo de Carga (`dataset_loader.py`)

Definimos una clase utilitaria para encapsular la conexión con la API de Hugging Face Datasets. Esto nos permite abstraer la lógica de descarga, gestionar posibles excepciones de red y retornar directamente un objeto `DataFrame` de Pandas listo para ser manipulado.

In [2]:
%%writefile dataset_loader.py
import pandas as pd
from datasets import load_dataset
from typing import Optional

class HFDatasetLoader:
    def __init__(self, dataset_name: str, split: Optional[str] = "train"):
        self.dataset_name = dataset_name
        self.split = split

    def load_as_dataframe(self) -> pd.DataFrame:
        try:
            dataset = load_dataset(self.dataset_name, split=self.split)
            return dataset.to_pandas()
        except Exception as e:
            raise RuntimeError(f"Error al cargar el dataset '{self.dataset_name}': {e}")

Writing dataset_loader.py


### Verificación de la Carga de Datos

Instanciamos el cargador configurado para el dataset de clasificación de Pokémon y realizamos una inspección visual de las primeras filas de la tabla cargada en memoria.

In [3]:
import importlib
import dataset_loader
# Forzar la recarga del módulo para aplicar cambios recientes
importlib.reload(dataset_loader)
from dataset_loader import HFDatasetLoader

# 1. Instanciar la clase
loader = HFDatasetLoader(dataset_name="JJMack/pokemon-classification-gen1-9", split="train")

# 2. Cargar DataFrame
df = loader.load_as_dataframe()

# 3. Visualizar los resultados
df.head()

c:\Users\Usuario\OneDrive\Escritorio\U\Analitica de datos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Usuario\OneDrive\Escritorio\U\Analitica de datos\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\datasets--JJMack--pokemon-classification-gen1-9. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Dev

,file_name,image_data,label,name,generation,Type 1,Type 2,HP,Attack,Defense,Sp.Attack,Sp.Defense,Speed,shiny
0,0044-Gloom-RG.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,44,Gloom,1,Grass,Poison,60,65,70,85,75,40,no
1,0069-Bellsprout-SpriteBack-3.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,69,Bellsprout,1,Grass,Poison,50,75,35,70,30,40,no
2,0529-Drilbur-MS8.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,529,Drilbur,5,Ground,NaN,60,85,40,30,45,68,no
3,0030-Nidorina-SpriteBack-2.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,30,Nidorina,1,Poison,NaN,70,62,67,55,55,56,no
4,0055-Golduck-BT.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,55,Golduck,1,Water,NaN,80,82,78,95,80,85,no


## 2. Módulo de Diagnóstico (`dataset_summarizer.py`)

Con el fin de automatizar el análisis exploratorio inicial, implementamos una clase que evalúa las dimensiones del dataset, identifica tipos de datos y cuantifica la presencia de valores nulos. Adicionalmente, genera resúmenes estadísticos diferenciando variables numéricas de categóricas, exportando un reporte detallado en formato de texto plano.

In [4]:
%%writefile dataset_summarizer.py
import pandas as pd
import io

class DatasetSummarizer:
    def __init__(self, dataframe: pd.DataFrame):
        if not isinstance(dataframe, pd.DataFrame):
            raise ValueError("El objeto proporcionado debe ser un DataFrame.")
        self.df = dataframe

    def generate_summary(self) -> str:
        buffer = io.StringIO()

        buffer.write("==================================================\n")
        buffer.write("              RESUMEN DEL DATAFRAME               \n")
        buffer.write("==================================================\n\n")

        rows, cols = self.df.shape
        buffer.write("--- 1. DIMENSIONES ---\n")
        buffer.write(f"Cantidad de Filas:    {rows}\n")
        buffer.write(f"Cantidad de Columnas: {cols}\n\n")

        buffer.write("--- 2. ESTRUCTURA DE COLUMNAS Y VALORES NULOS ---\n")
        buffer.write(f"{'Nombre de Columna':<35} | {'Tipo de Dato':<12} | {'Valores Nulos':<13} | {'% Nulos':<8}\n")
        buffer.write("-" * 77 + "\n")

        for col in self.df.columns:
            dtype = str(self.df[col].dtype)
            null_count = self.df[col].isnull().sum()
            null_pct = (null_count / rows) * 100 if rows > 0 else 0
            buffer.write(f"{col:<35} | {dtype:<12} | {null_count:<13} | {null_pct:<7.2f}%\n")
        buffer.write("\n")

        buffer.write("--- 3. ESTADÍSTICAS DESCRIPTIVAS ---\n")
        num_df = self.df.select_dtypes(include=['number'])
        if not num_df.empty:
            buffer.write("\n[Variables Numéricas]\n")
            buffer.write(num_df.describe().T.to_string())
            buffer.write("\n\n")

        cat_df = self.df.select_dtypes(exclude=['number'])
        if not cat_df.empty:
            buffer.write("[Variables Categóricas / Texto]\n")
            buffer.write(cat_df.describe().T.to_string())
            buffer.write("\n\n")

        buffer.write("==================================================\n")
        buffer.write("               FIN DEL INFORME                    \n")
        buffer.write("==================================================\n")

        return buffer.getvalue()

    def save_summary_to_file(self, filename: str = "resumen_dataset.txt") -> str:
        summary_text = self.generate_summary()
        with open(filename, "w", encoding="utf-8") as f:
            f.write(summary_text)
        print(f"Resumen guardado exitosamente en '{filename}'")
        return filename

Writing dataset_summarizer.py


### Ejecución del Diagnóstico y Exportación de Reporte

Corremos el resumidor sobre nuestro conjunto de datos actual para validar la integridad de las columnas y guardar el reporte `resumen_pokemon.txt` en el entorno local.

In [5]:
from dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

# 1. Cargar datos usando la clase del punto 2
loader = HFDatasetLoader(dataset_name="JJMack/pokemon-classification-gen1-9", split="train")
df = loader.load_as_dataframe()

# 2. Resumir datos y guardar en .txt usando la clase del punto 3
summarizer = DatasetSummarizer(df)
summarizer.save_summary_to_file("resumen_pokemon.txt")

Resumen guardado exitosamente en 'resumen_pokemon.txt'


'resumen_pokemon.txt'

## 3. Transformación y Limpieza de Datos

En esta sección se aplican las reglas de negocio necesarias para estructurar los datos finales. El proceso incluye:
1. Reestructuración de nombres de columnas clave.
2. Remoción de registros inconsistentes o sin etiquetas válidas.
3. Generación de metadatos de procesamiento (`processed_status`).
4. Normalización de variables numéricas a formato categórico de texto para análisis estadísticos posteriores.

In [6]:
# Usar clase del punto 2 con el dataset de Pokémon de Hugging Face
loader = HFDatasetLoader(dataset_name="JJMack/pokemon-classification-gen1-9", split="train")
df_original = loader.load_as_dataframe()

print(f"Dimensiones originales: {df_original.shape}")
df_original.head()

Dimensiones originales: (13188, 14)


,file_name,image_data,label,name,generation,Type 1,Type 2,HP,Attack,Defense,Sp.Attack,Sp.Defense,Speed,shiny
0,0044-Gloom-RG.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,44,Gloom,1,Grass,Poison,60,65,70,85,75,40,no
1,0069-Bellsprout-SpriteBack-3.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,69,Bellsprout,1,Grass,Poison,50,75,35,70,30,40,no
2,0529-Drilbur-MS8.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,529,Drilbur,5,Ground,NaN,60,85,40,30,45,68,no
3,0030-Nidorina-SpriteBack-2.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,30,Nidorina,1,Poison,NaN,70,62,67,55,55,56,no
4,0055-Golduck-BT.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,55,Golduck,1,Water,NaN,80,82,78,95,80,85,no


### Aplicación del Pipeline de Transformación

In [7]:
# Crear una copia
df_transformed = df_original.copy()

# Transformación 1: Renombrar las columnas principales (por ejemplo, 'label' a 'pokemon_label')
if 'label' in df_transformed.columns:
    df_transformed = df_transformed.rename(columns={'label': 'pokemon_label'})

# Transformación 2: Filtrar filas (si existe una columna de categoría o índice, o seleccionando un subconjunto de registros de prueba)
# Por ejemplo, si queremos limitar o filtrar registros válidos que no tengan valores nulos en la etiqueta:
if 'pokemon_label' in df_transformed.columns:
    df_transformed = df_transformed.dropna(subset=['pokemon_label'])

# Transformación 3: Crear una nueva columna indicando si la ruta/imagen está disponible o creando una marca de procesamiento
df_transformed['processed_status'] = "Verified"

# Transformación 4: Convertir las etiquetas a tipo texto o categoría estándar para facilitar el resumen estadístico
if 'pokemon_label' in df_transformed.columns:
    df_transformed['pokemon_label_str'] = df_transformed['pokemon_label'].astype(str)

print(f"Dimensiones después de las transformaciones: {df_transformed.shape}")
df_transformed.head()

Dimensiones después de las transformaciones: (13188, 16)


,file_name,image_data,pokemon_label,name,generation,Type 1,Type 2,HP,Attack,Defense,Sp.Attack,Sp.Defense,Speed,shiny,processed_status,pokemon_label_str
0,0044-Gloom-RG.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,44,Gloom,1,Grass,Poison,60,65,70,85,75,40,no,Verified,44
1,0069-Bellsprout-SpriteBack-3.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,69,Bellsprout,1,Grass,Poison,50,75,35,70,30,40,no,Verified,69
2,0529-Drilbur-MS8.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,529,Drilbur,5,Ground,NaN,60,85,40,30,45,68,no,Verified,529
3,0030-Nidorina-SpriteBack-2.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,30,Nidorina,1,Poison,NaN,70,62,67,55,55,56,no,Verified,30
4,0055-Golduck-BT.png,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,55,Golduck,1,Water,NaN,80,82,78,95,80,85,no,Verified,55
